## SRP261402 / GSE150461

**paper:** [PMID: 32485163](https://www.sciencedirect.com/science/article/pii/S0003269720303134) - Identification and characterization of microRNAs (miRNAs) and their transposable element origins in the saltwater crocodile, Crocodylus porosus, 2020

**date, curator:** 2026-03-04, Sara Carsanaro

**notes**
* they call these hatchlings, however around 1 year is the transition from hatchling to juvenile, so i think sexually immature stage makes the most sense since they are approx. 359 - 373 days old
* unfortunately i cannot determine which samples are from runts and which samples are from non-runts


### annotation summary

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,testis,UBERON:0000473,testis,perfect match
3,brain,UBERON:0000955,brain,perfect match
7,heart,UBERON:0000948,heart,perfect match
11,stomach,UBERON:0000945,stomach,perfect match
19,liver,UBERON:0002107,liver,perfect match
27,small intestine,UBERON:0002108,small intestine,perfect match
31,tongue,UBERON:0001723,tongue,perfect match
35,spleen,UBERON:0002106,spleen,perfect match
39,cloaca,UBERON:0000162,cloaca,perfect match
43,jaw skin,UBERON:1000015,skin of snout,perfect match


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,"hatchling, approximately 359 - 373 days old",UBERON:0000112,sexually immature stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP261402"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes
Fetching RunIds and ReadHashes for each library...can take several minutes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX8335641,SRP261402,Illumina HiSeq 2500,SRS6654635,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550896,testis,"hatchling, approximately 359 - 373 days old",,,,M,,,8502,,,,,,Testis_D2 Sample 3,"SAMN14910677,GSM4550896",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783132,58D964C9AABC268238DA652FF4A87DBD
1,SRX8335640,SRP261402,Illumina HiSeq 2500,SRS6654634,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550895,testis,"hatchling, approximately 359 - 373 days old",,,,M,,,8502,,,,,,Testis_D2 Sample 2,"SAMN14910678,GSM4550895",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783131,FDDB119E49C4C30A7F0472266BCD9231
2,SRX8335639,SRP261402,Illumina HiSeq 2500,SRS6654633,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550894,testis,"hatchling, approximately 359 - 373 days old",,,,M,,,8502,,,,,,Testis_D2 Sample 1,"SAMN14910679,GSM4550894",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783130,C713E55C70B5F8127F81C387750D7294
3,SRX8335638,SRP261402,Illumina HiSeq 2500,SRS6654632,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550953,brain,"hatchling, approximately 359 - 373 days old",,,,NA,,,8502,,,,,,Brain_Q3 Sample 60,"SAMN14910680,GSM4550953",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling Brain cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783189,9BB03A75252A603C80A84909F7573532
4,SRX8335637,SRP261402,Illumina HiSeq 2500,SRS6654631,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550952,brain,"hatchling, approximately 359 - 373 days old",,,,NA,,,8502,,,,,,Brain_Q3 Sample 59,"SAMN14910681,GSM4550952",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Ha

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['brain' 'cloaca' 'heart' 'jaw skin' 'liver' 'small intestine' 'spleen'
 'stomach' 'testis' 'tongue']


In [6]:
# brain
library.loc[library["infoOrgan"] == "brain", "anatId"] = "UBERON:0000955"
library.loc[library["infoOrgan"] == "brain", "anatName"] = "brain"

# cloaca
library.loc[library["infoOrgan"] == "cloaca", "anatId"] = "UBERON:0000162"
library.loc[library["infoOrgan"] == "cloaca", "anatName"] = "cloaca"

# heart
library.loc[library["infoOrgan"] == "heart", "anatId"] = "UBERON:0000948"
library.loc[library["infoOrgan"] == "heart", "anatName"] = "heart"

# jaw skin
library.loc[library["infoOrgan"] == "jaw skin", "anatId"] = "UBERON:1000015"
library.loc[library["infoOrgan"] == "jaw skin", "anatName"] = "skin of snout"

# liver
library.loc[library["infoOrgan"] == "liver", "anatId"] = "UBERON:0002107"
library.loc[library["infoOrgan"] == "liver", "anatName"] = "liver"

# small intestine
library.loc[library["infoOrgan"] == "small intestine", "anatId"] = "UBERON:0002108"
library.loc[library["infoOrgan"] == "small intestine", "anatName"] = "small intestine"

# spleen
library.loc[library["infoOrgan"] == "spleen", "anatId"] = "UBERON:0002106"
library.loc[library["infoOrgan"] == "spleen", "anatName"] = "spleen"

# stomach
library.loc[library["infoOrgan"] == "stomach", "anatId"] = "UBERON:0000945"
library.loc[library["infoOrgan"] == "stomach", "anatName"] = "stomach"

# testis
library.loc[library["infoOrgan"] == "testis", "anatId"] = "UBERON:0000473"
library.loc[library["infoOrgan"] == "testis", "anatName"] = "testis"

# tongue
library.loc[library["infoOrgan"] == "tongue", "anatId"] = "UBERON:0001723"
library.loc[library["infoOrgan"] == "tongue", "anatName"] = "tongue"

# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX8335641,SRP261402,Illumina HiSeq 2500,SRS6654635,UBERON:0000473,testis,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550896,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,,M,,,8502,,,,,,Testis_D2 Sample 3,"SAMN14910677,GSM4550896",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783132,58D964C9AABC268238DA652FF4A87DBD
1,SRX8335640,SRP261402,Illumina HiSeq 2500,SRS6654634,UBERON:0000473,testis,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550895,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,,M,,,8502,,,,,,Testis_D2 Sample 2,"SAMN14910678,GSM4550895",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783131,FDDB119E49C4C30A7F0472266BCD9231
2,SRX8335639,SRP261402,Illumina HiSeq 2500,SRS6654633,UBERON:0000473,testis,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550894,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,,M,,,8502,,,,,,Testis_D2 Sample 1,"SAMN14910679,GSM4550894",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783130,C713E55C70B5F8127F81C387750D7294
3,SRX8335638,SRP261402,Illumina HiSeq 2500,SRS6654632,UBERON:0000955,brain,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550953,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,,NA,,,8502,,,,,,Brain_Q3 Sample 60,"SAMN14910680,GSM4550953",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling Brain cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783189,9BB03A75252A603C80A84909F7573532
4,SRX8335637,SRP261402,Illumina HiSeq 2500,SRS6654631,UBERON:0000955,brain,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550952,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,,NA,,,8502,,,,,,Brain_Q3 Sample 59,"SAMN14910681,GSM4550952",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated us

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['hatchling, approximately 359 - 373 days old']


In [8]:
# they call these hatchlings, however around 1 year is the transition from hatchling to juvenile, so i think sexually immature makes more sense here
library.loc[:,'stageId'] = 'UBERON:0000112'
library.loc[:,'stageName'] = 'sexually immature stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'other'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX8335641,SRP261402,Illumina HiSeq 2500,SRS6654635,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550896,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,,,,,,Testis_D2 Sample 3,"SAMN14910677,GSM4550896",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783132,58D964C9AABC268238DA652FF4A87DBD
1,SRX8335640,SRP261402,Illumina HiSeq 2500,SRS6654634,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550895,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,,,,,,Testis_D2 Sample 2,"SAMN14910678,GSM4550895",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783131,FDDB119E49C4C30A7F0472266BCD9231
2,SRX8335639,SRP261402,Illumina HiSeq 2500,SRS6654633,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550894,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,,,,,,Testis_D2 Sample 1,"SAMN14910679,GSM4550894",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783130,C713E55C70B5F8127F81C387750D7294
3,SRX8335638,SRP261402,Illumina HiSeq 2500,SRS6654632,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550953,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,NA,,,8502,,,,,,Brain_Q3 Sample 60,"SAMN14910680,GSM4550953",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling Brain cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783189,9BB03A75252A603C80A84909F7573532
4,SRX8335637,SRP261402,Illumina HiSeq 2500,SRS6654631,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550952,brain,"hatchling, approximately 359 - 373 days old",perfect match,not doc

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Small RNA Library Prep Set'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'miRNA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX8335641,SRP261402,Illumina HiSeq 2500,SRS6654635,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550896,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 3,"SAMN14910677,GSM4550896",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783132,58D964C9AABC268238DA652FF4A87DBD
1,SRX8335640,SRP261402,Illumina HiSeq 2500,SRS6654634,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550895,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 2,"SAMN14910678,GSM4550895",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783131,FDDB119E49C4C30A7F0472266BCD9231
2,SRX8335639,SRP261402,Illumina HiSeq 2500,SRS6654633,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550894,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 1,"SAMN14910679,GSM4550894",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783130,C713E55C70B5F8127F81C387750D7294
3,SRX8335638,SRP261402,Illumina HiSeq 2500,SRS6654632,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550953,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,NA,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Brain_Q3 Sample 60,"SAMN14910680,GSM4550953",,,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling Brain cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783189,9BB03A75252A603C80A84909F7573532
4,SRX8335637,SRP261402,Illumina HiSeq 2500,SRS665

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [11]:
library.loc[:,'sampleAge_value'] = '359 - 373'
library.loc[:,'sampleAge_unit'] = 'day'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX8335641,SRP261402,Illumina HiSeq 2500,SRS6654635,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550896,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 3,"SAMN14910677,GSM4550896",359 - 373,day,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783132,58D964C9AABC268238DA652FF4A87DBD
1,SRX8335640,SRP261402,Illumina HiSeq 2500,SRS6654634,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550895,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 2,"SAMN14910678,GSM4550895",359 - 373,day,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783131,FDDB119E49C4C30A7F0472266BCD9231
2,SRX8335639,SRP261402,Illumina HiSeq 2500,SRS6654633,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550894,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 1,"SAMN14910679,GSM4550894",359 - 373,day,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783130,C713E55C70B5F8127F81C387750D7294
3,SRX8335638,SRP261402,Illumina HiSeq 2500,SRS6654632,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550953,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,NA,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Brain_Q3 Sample 60,"SAMN14910680,GSM4550953",359 - 373,day,,,,,,,,,04/03/2026,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling Brain cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783189,9BB03A75252A603C80A84909F7573532
4

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-05'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX8335641,SRP261402,Illumina HiSeq 2500,SRS6654635,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550896,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 3,"SAMN14910677,GSM4550896",359 - 373,day,,,,,,,,SAC,2026-03-05,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783132,58D964C9AABC268238DA652FF4A87DBD
1,SRX8335640,SRP261402,Illumina HiSeq 2500,SRS6654634,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550895,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 2,"SAMN14910678,GSM4550895",359 - 373,day,,,,,,,,SAC,2026-03-05,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783131,FDDB119E49C4C30A7F0472266BCD9231
2,SRX8335639,SRP261402,Illumina HiSeq 2500,SRS6654633,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550894,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 1,"SAMN14910679,GSM4550894",359 - 373,day,,,,,,,,SAC,2026-03-05,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling testis cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783130,C713E55C70B5F8127F81C387750D7294
3,SRX8335638,SRP261402,Illumina HiSeq 2500,SRS6654632,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550953,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,NA,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Brain_Q3 Sample 60,"SAMN14910680,GSM4550953",359 - 373,day,,,,,,,,SAC,2026-03-05,"Tissue types were flash frozen in liquid nitrogen. Tissue types were stored in RNAlater. Total RNA was isolated using Trizol, RNeasy Mini Kit (Qiagen) and the Directt-Zol RNA Miniprep Kit (Zymo Research Inc.). All libraries were constructed using NEBNext Small RNA Library Prep for Illumina (NEB Inc.) Small RNA sequencing (Deep sequencing)",,,,Hatchling Brain cells,,,,TRANSCRIPTOMIC,size fractionation,SRR11783189,9BB03A75252A603C80A849

#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 32485163, Runt hatchlings 359 days, Normal hatchlings 373 days, not clear which samples are which'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX8335641,SRP261402,Illumina HiSeq 2500,SRS6654635,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550896,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 3,"SAMN14910677,GSM4550896",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Normal hatchlings 373 days, not clear which samples are which",,,SAC,2026-03-05
1,SRX8335640,SRP261402,Illumina HiSeq 2500,SRS6654634,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550895,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 2,"SAMN14910678,GSM4550895",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Normal hatchlings 373 days, not clear which samples are which",,,SAC,2026-03-05
2,SRX8335639,SRP261402,Illumina HiSeq 2500,SRS6654633,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550894,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 1,"SAMN14910679,GSM4550894",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Normal hatchlings 373 days, not clear which samples are which",,,SAC,2026-03-05
3,SRX8335638,SRP261402,Illumina HiSeq 2500,SRS6654632,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550953,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,NA,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Brain_Q3 Sample 60,"SAMN14910680,GSM4550953",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Normal hatchlings 373 days, not clear which samples are which",,,SAC,2026-03-05
4,SRX8335637,SRP261402,Illumina HiSeq 2500,SRS6654631,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550952,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,NA,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Brain_Q3 Sample 59,"SAMN14910681,GSM4550952",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Normal hatchlings 373 days, not clear which samples are which",,,SAC,2026-03-05
5,SRX8335636,SRP261402,Illumina HiSeq 2500,SRS6654630,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550951,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,NA,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Brain_Q3 Sample 58,"SAMN14910683,GSM4550951",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Normal hatchlings 373 days, not clear which samples are which",,,SAC,2026-03-05
6,SRX8335635,SRP261402,Illumina HiSeq 2500,SRS6654629,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4550950,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,NA,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Brain_Q3 Sample 57,"SAMN14910636,GSM4550950",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Normal hatchlings 373 days, not clear which samples are which",,,SAC,

### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP261402,"Identification and characterization of microRNAs (miRNAs) and their transposable element origins in the saltwater crocodile, Crocodylus porosus","MicroRNAs (miRNAs) are 18-24 nucleotide autonomous regulatory RNA molecules found in all eukaryotes. They are involved in the regulation of a multitude of genetic and biological pathways through post transcriptional gene silencing and/or translational repression. Previous data has suggested a slow evolutionary rate for the saltwater crocodile (Crocodylus porosus) over the past several million years when compared to its closest extant relatives, the birds. Understanding genome regulation, adaptive capabilities and physiological features in the saltwater crocodile in the context of relatively slow genomic change thus holds significant potential for the investigation of genomics, evolution and adaptive studies. Utilizing eleven different tissue types and sixteen small RNA libraries, we report a catalog of 644 miRNAs in the saltwater crocodile with > 78% of miRNAs being potentially novel to crocodilians. We also predicted and identified targets for the miRNAs as well as analyzed the relationship of the miRNA repertoire to transposable elements (TEs) in the saltwater crocodile that showed an increased association of DNA transposons with miRNA biogenesis when compared to retrotransposons. Phylogenetic analysis of C. porosus miRNA expectedly revealed highest number of miRNAs in sister crocodilian clades of the American Alligator and the Indian Gharial. This work reports the first comprehensive analysis of miRNAs in Crocodylus porosus for and addresses the potential impacts of miRNAs in regulating the genome in the saltwater crocodile as well as supporting the role of TEs as a source for miRNAs, adding to the increasing evidence that TEs play a significant role in the evolution of gene regulation. Overall design: Construction of Illumina Deep sequening small RNA libraries in 11 different tissue typoes in the C porosus and bioinformatic analysis to predict C porosus known and novel miRNAs, their targets, expression profiles and other genomic features of the miRNAs in the C porosus",SRA,,,,,,GSE150461,PRJNA632490,32485163,,10.1016/j.ab.2020.113781,,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

60

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP261402,"Identification and characterization of microRNAs (miRNAs) and their transposable element origins in the saltwater crocodile, Crocodylus porosus","MicroRNAs (miRNAs) are 18-24 nucleotide autonomous regulatory RNA molecules found in all eukaryotes. They are involved in the regulation of a multitude of genetic and biological pathways through post transcriptional gene silencing and/or translational repression. Previous data has suggested a slow evolutionary rate for the saltwater crocodile (Crocodylus porosus) over the past several million years when compared to its closest extant relatives, the birds. Understanding genome regulation, adaptive capabilities and physiological features in the saltwater crocodile in the context of relatively slow genomic change thus holds significant potential for the investigation of genomics, evolution and adaptive studies. Utilizing eleven different tissue types and sixteen small RNA libraries, we report a catalog of 644 miRNAs in the saltwater crocodile with > 78% of miRNAs being potentially novel to crocodilians. We also predicted and identified targets for the miRNAs as well as analyzed the relationship of the miRNA repertoire to transposable elements (TEs) in the saltwater crocodile that showed an increased association of DNA transposons with miRNA biogenesis when compared to retrotransposons. Phylogenetic analysis of C. porosus miRNA expectedly revealed highest number of miRNAs in sister crocodilian clades of the American Alligator and the Indian Gharial. This work reports the first comprehensive analysis of miRNAs in Crocodylus porosus for and addresses the potential impacts of miRNAs in regulating the genome in the saltwater crocodile as well as supporting the role of TEs as a source for miRNAs, adding to the increasing evidence that TEs play a significant role in the evolution of gene regulation. Overall design: Construction of Illumina Deep sequening small RNA libraries in 11 different tissue typoes in the C porosus and bioinformatic analysis to predict C porosus known and novel miRNAs, their targets, expression profiles and other genomic features of the miRNAs in the C porosus",SRA,total,Bgee 1K,60,NEBNext Small RNA Library Prep Set,full_length,GSE150461,PRJNA632490,32485163,,10.1016/j.ab.2020.113781,,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://www.sciencedirect.com/science/article/pii/S0003269720303134'
#experiment.loc[:,'DOI'] = ''
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP261402,"Identification and characterization of microRNAs (miRNAs) and their transposable element origins in the saltwater crocodile, Crocodylus porosus","MicroRNAs (miRNAs) are 18-24 nucleotide autonomous regulatory RNA molecules found in all eukaryotes. They are involved in the regulation of a multitude of genetic and biological pathways through post transcriptional gene silencing and/or translational repression. Previous data has suggested a slow evolutionary rate for the saltwater crocodile (Crocodylus porosus) over the past several million years when compared to its closest extant relatives, the birds. Understanding genome regulation, adaptive capabilities and physiological features in the saltwater crocodile in the context of relatively slow genomic change thus holds significant potential for the investigation of genomics, evolution and adaptive studies. Utilizing eleven different tissue types and sixteen small RNA libraries, we report a catalog of 644 miRNAs in the saltwater crocodile with > 78% of miRNAs being potentially novel to crocodilians. We also predicted and identified targets for the miRNAs as well as analyzed the relationship of the miRNA repertoire to transposable elements (TEs) in the saltwater crocodile that showed an increased association of DNA transposons with miRNA biogenesis when compared to retrotransposons. Phylogenetic analysis of C. porosus miRNA expectedly revealed highest number of miRNAs in sister crocodilian clades of the American Alligator and the Indian Gharial. This work reports the first comprehensive analysis of miRNAs in Crocodylus porosus for and addresses the potential impacts of miRNAs in regulating the genome in the saltwater crocodile as well as supporting the role of TEs as a source for miRNAs, adding to the increasing evidence that TEs play a significant role in the evolution of gene regulation. Overall design: Construction of Illumina Deep sequening small RNA libraries in 11 different tissue typoes in the C porosus and bioinformatic analysis to predict C porosus known and novel miRNAs, their targets, expression profiles and other genomic features of the miRNAs in the C porosus",SRA,total,Bgee 1K,60,NEBNext Small RNA Library Prep Set,full_length,GSE150461,PRJNA632490,32485163,https://www.sciencedirect.com/science/article/pii/S0003269720303134,10.1016/j.ab.2020.113781,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
55519,SRX1168114,SRP062962,Illumina HiSeq 2000,SRS1047932,UBERON:0002548,larva,UBERON:0000069,larval stage,,whole larvae,First feeding,perfect match,not documented,other,NA,NA,,8175,TruSeq RNA Sample Preparation Kit,full_length,polyA,,17,Sample from Sparus aurata,SAMN04013652,,,,,,,"PMID:27461489, For transcriptome analysis thre...",,,ANN,2026-03-03
55520,SRX1168113,SRP062962,Illumina HiSeq 2000,SRS1047932,UBERON:0002548,larva,UBERON:0000069,larval stage,,whole larvae,First feeding,perfect match,not documented,other,NA,NA,,8175,TruSeq RNA Sample Preparation Kit,full_length,polyA,,17,Sample from Sparus aurata,SAMN04013652,,,,,,,"PMID:27461489, For transcriptome analysis thre...",,,ANN,2026-03-03
55521,SRX8335641,SRP261402,Illumina HiSeq 2500,SRS6654635,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 3,"SAMN14910677,GSM4550896",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Norm...",,,SAC,2026-03-05
55522,SRX8335640,SRP261402,Illumina HiSeq 2500,SRS6654634,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 2,"SAMN14910678,GSM4550895",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Norm...",,,SAC,2026-03-05
55523,SRX8335639,SRP261402,Illumina HiSeq 2500,SRS6654633,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,testis,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,M,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Testis_D2 Sample 1,"SAMN14910679,GSM4550894",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Norm...",,,SAC,2026-03-05
55524,SRX8335638,SRP261402,Illumina HiSeq 2500,SRS6654632,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,NA,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Brain_Q3 Sample 60,"SAMN14910680,GSM4550953",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Norm...",,,SAC,2026-03-05
55525,SRX8335637,SRP261402,Illumina HiSeq 2500,SRS6654631,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,brain,"hatchling, approximately 359 - 373 days old",perfect match,not documented,other,NA,,,8502,NEBNext Small RNA Library Prep Set,full_length,miRNA,,,Brain_Q3 Sample 59,"SAMN14910681,GSM4550952",359 - 373,day,,,,,"PMID: 32485163, Runt hatchlings 359 days, Norm...",,,SAC,2026-03-05


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1071,SRP361110,Transcriptomics of Parental Care in the Hypoth...,-,SRA,total,Bgee 1K,12,,,,PRJNA808854,35269661,https://pmc.ncbi.nlm.nih.gov/articles/PMC8910180/,10.3390/ijms23052518,,
1072,SRP062962,Sparus aurata Transcriptome or Gene expression,Expression Patterns After Early Life Stress In...,SRA,total,Bgee 1K,51,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA294043,27461489,https://pmc.ncbi.nlm.nih.gov/articles/PMC4962366/,10.1186/s12864-016-2874-0,,"Bgee curator notes: complex experiment schema,..."
1073,SRP261402,Identification and characterization of microRN...,MicroRNAs (miRNAs) are 18-24 nucleotide autono...,SRA,total,Bgee 1K,60,NEBNext Small RNA Library Prep Set,full_length,GSE150461,PRJNA632490,32485163,https://www.sciencedirect.com/science/article/...,10.1016/j.ab.2020.113781,,


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [28]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [29]:
! git add $git_experiment_path $git_library_path

In [30]:
! git commit -m $commit_message_exp

[develop 917ecf2] adding annotated bulk experiment SRP261402
 2 files changed, 61 insertions(+)


In [31]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 4.52 KiB | 1.13 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   ef90c7b..917ecf2  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push